# Constraint Satisfaction Problems (CSP) — Tutorial Completo
## Colorear el Mapa de Australia

**Basado en:** Russell & Norvig, AIMA Cap. 6 | Glouppe INFO8006

Implementamos: Backtracking, MRV, LCV, Forward Checking, AC-3, Min-Conflicts

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from collections import deque
import copy, random

## 1. Definición del CSP

- **Variables** X = {WA, NT, Q, NSW, V, SA, T}
- **Dominios** Di = {rojo, verde, azul}
- **Restricciones** C = {regiones adyacentes ≠ mismo color}

In [ ]:
variables = ["WA", "NT", "Q", "NSW", "V", "SA", "T"]
domains = {v: ["rojo", "verde", "azul"] for v in variables}
constraints = [("WA","NT"),("WA","SA"),("NT","SA"),("NT","Q"),
               ("SA","Q"),("SA","NSW"),("SA","V"),("Q","NSW"),("NSW","V")]
neighbors = {v: [] for v in variables}
for a, b in constraints:
    neighbors[a].append(b)
    neighbors[b].append(a)

print("Variables:", variables)
print(f"Restricciones: {len(constraints)} pares")
for a,b in constraints: print(f"  {a} ≠ {b}")
print("
Vecinos (SA tiene 5 — la más restringida):")
for v in variables: print(f"  {v}: {neighbors[v]}")

### 1.1 Grafo de Restricciones
Nodos=variables, aristas=restricciones. T está aislada (subproblema independiente).

In [ ]:
def plot_graph(assignment=None, title=""):
    G = nx.Graph(); G.add_nodes_from(variables); G.add_edges_from(constraints)
    pos = {"WA":(0,1),"NT":(1,2),"SA":(1,0.8),"Q":(2,2),"NSW":(2.2,1),"V":(1.8,0),"T":(1.8,-1)}
    cmap = {"rojo":"#E24B4A","verde":"#1D9E75","azul":"#378ADD"}
    cols = [cmap.get(assignment.get(v,""),"#D3D1C7") if assignment else "#D3D1C7" for v in G.nodes()]
    fig,ax = plt.subplots(figsize=(8,6))
    nx.draw(G,pos,with_labels=True,node_color=cols,node_size=1800,font_size=12,
            font_weight="bold",edge_color="#888",width=2,ax=ax,edgecolors="#444",linewidths=2)
    ax.set_title(title or "Grafo de Restricciones",fontsize=14,fontweight="bold")
    plt.tight_layout(); plt.show()

plot_graph()

## 2. Verificación de Consistencia
¿Asignar var=value viola alguna restricción con vecinos ya asignados?

In [ ]:
def is_consistent(var, value, assignment):
    for neighbor in neighbors[var]:
        if neighbor in assignment and assignment[neighbor] == value:
            return False
    return True

test = {"WA": "rojo", "NT": "verde"}
print("Asignación:", test)
print(f"  ¿SA=rojo? {is_consistent('SA','rojo',test)} (No: WA=rojo es vecino)")
print(f"  ¿SA=azul? {is_consistent('SA','azul',test)} (Sí: azul≠rojo, azul≠verde)")

## 3. Backtracking Básico
DFS que asigna una variable por nivel. Retrocede al detectar violación.
- SELECT-UNASSIGNED-VARIABLE: orden fijo
- ORDER-DOMAIN-VALUES: orden natural
- Sin propagación de restricciones

In [ ]:
def backtracking_basic():
    nodes = [0]
    def backtrack(assignment):
        if len(assignment) == len(variables): return dict(assignment)
        var = next(v for v in variables if v not in assignment)
        for value in domains[var]:
            nodes[0] += 1
            if is_consistent(var, value, assignment):
                assignment[var] = value
                result = backtrack(assignment)
                if result is not None: return result
                del assignment[var]
        return None
    return backtrack({}), nodes[0]

sol, n = backtracking_basic()
print("Backtracking básico:")
for v in variables: print(f"  {v} = {sol[v]}")
print(f"Nodos explorados: {n}")
plot_graph(sol, f"Backtracking básico ({n} nodos)")

## 4. Heurísticas: MRV + LCV

**MRV (Minimum Remaining Values)**: asignar primero la variable con menos valores legales. "Fail-first" — detectar fallos temprano.

**LCV (Least Constraining Value)**: probar primero el valor que deja más opciones a los vecinos. "Fail-last" — maximizar chances de éxito.

**¿Por qué la asimetría?** Variables: hay que asignar TODAS → fallar rápido. Valores: solo necesitamos UNO que funcione → probar el más prometedor.

In [ ]:
def backtracking_mrv_lcv():
    nodes = [0]
    def backtrack(assignment):
        if len(assignment) == len(variables): return dict(assignment)
        # MRV: variable con menos valores legales
        unassigned = [v for v in variables if v not in assignment]
        var = min(unassigned, key=lambda v:
                  sum(1 for val in domains[v] if is_consistent(v,val,assignment)))
        # LCV: valor que menos restringe vecinos
        def conflicts(val):
            return sum(1 for n in neighbors[var] if n not in assignment
                       for nv in domains[n] if nv == val)
        for value in sorted(domains[var], key=conflicts):
            nodes[0] += 1
            if is_consistent(var, value, assignment):
                assignment[var] = value
                result = backtrack(assignment)
                if result is not None: return result
                del assignment[var]
        return None
    return backtrack({}), nodes[0]

sol2, n2 = backtracking_mrv_lcv()
print("Backtracking + MRV + LCV:")
for v in variables: print(f"  {v} = {sol2[v]}")
print(f"Nodos explorados: {n2}")

## 5. Forward Checking

Al asignar var=value, **eliminar** value del dominio de todos los vecinos no asignados. Si algún dominio queda vacío → retroceder inmediatamente.

Es como "mirar un paso adelante" para detectar fallos antes de que ocurran.

In [ ]:
def backtracking_fc():
    nodes = [0]
    def backtrack(assignment, curr_domains):
        if len(assignment) == len(variables): return dict(assignment)
        # MRV
        unassigned = [v for v in variables if v not in assignment]
        var = min(unassigned, key=lambda v: len(curr_domains[v]))
        for value in list(curr_domains[var]):
            nodes[0] += 1
            if is_consistent(var, value, assignment):
                assignment[var] = value
                # Forward check: propagar
                saved = {}
                ok = True
                for nb in neighbors[var]:
                    if nb not in assignment and value in curr_domains[nb]:
                        saved[nb] = list(curr_domains[nb])
                        curr_domains[nb].remove(value)
                        if not curr_domains[nb]: ok = False; break
                if ok:
                    result = backtrack(assignment, curr_domains)
                    if result is not None: return result
                # Restaurar dominios
                for nb, sv in saved.items(): curr_domains[nb] = sv
                del assignment[var]
        return None
    cd = {v: list(d) for v,d in domains.items()}
    return backtrack({}, cd), nodes[0]

sol3, n3 = backtracking_fc()
print("Forward Checking + MRV:")
for v in variables: print(f"  {v} = {sol3[v]}")
print(f"Nodos explorados: {n3}")

## 6. AC-3 (Arc Consistency)

Un arco Xi→Xj es **arc-consistente** si para cada valor x en Di, existe al menos un valor y en Dj que satisface la restricción.

AC-3 hace TODOS los arcos consistentes propagando iterativamente.



In [ ]:
def revise(doms, xi, xj):
    """Elimina valores de Di sin soporte en Dj. Retorna True si eliminó algo."""
    revised = False
    for x in list(doms[xi]):
        if not any(y != x for y in doms[xj]):
            doms[xi].remove(x)
            revised = True
    return revised

def ac3(doms):
    """AC-3. Modifica dominios in-place. Retorna False si inconsistencia."""
    queue = deque()
    for a,b in constraints:
        queue.append((a,b)); queue.append((b,a))
    steps = 0
    while queue:
        xi, xj = queue.popleft(); steps += 1
        if revise(doms, xi, xj):
            if not doms[xi]: return False, steps
            for xk in neighbors[xi]:
                if xk != xj: queue.append((xk, xi))
    return True, steps

# Demo: AC-3 como preprocesamiento con 3 colores
d1 = {v: list(domains[v]) for v in variables}
ok, steps = ac3(d1)
print(f"AC-3 con 3 colores: {steps} revisiones")
print("Dominios:", {v:d1[v] for v in variables})
print("→ No poda nada (cada valor siempre tiene soporte)
")

# Demo: AC-3 detecta fallo con 2 colores + WA=rojo
d2 = {v: ["rojo","verde"] for v in variables}
d2["WA"] = ["rojo"]
print("AC-3 con 2 colores (WA=rojo fijo):")
ok2, steps2 = ac3(d2)
print(f"  Resultado: {'OK' if ok2 else 'FALLO'} en {steps2} revisiones")
for v in variables: print(f"  {v}: {d2[v]}")
if not ok2: print("→ AC-3 detectó que 2 colores son insuficientes!")

## 7. Min-Conflicts (Búsqueda Local)

Enfoque diferente: empieza con asignación completa aleatoria (posiblemente inválida) y mejora iterativamente.

1. Elegir variable con conflictos
2. Cambiar su valor al que minimice conflictos
3. Repetir

**Dato**: resuelve N-Queens con N=1,000,000 en ~50 pasos.

In [ ]:
def min_conflicts(max_steps=1000):
    assignment = {v: random.choice(domains[v]) for v in variables}
    for step in range(max_steps):
        conflicted = [v for v in variables
                      if any(assignment.get(n)==assignment[v] for n in neighbors[v])]
        if not conflicted: return assignment, step
        var = random.choice(conflicted)
        # Valor que minimiza conflictos
        best = min(domains[var],
                   key=lambda val: sum(1 for n in neighbors[var] if assignment.get(n)==val))
        assignment[var] = best
    return None, max_steps

random.seed(42)
print("Min-Conflicts — 10 ejecuciones:")
for i in range(10):
    sol_mc, steps = min_conflicts()
    if sol_mc:
        print(f"  Run {i+1}: solución en {steps} pasos")
    else:
        print(f"  Run {i+1}: no encontró solución")

## 8. Comparación de Estrategias

In [ ]:
results = {}
_, n1 = backtracking_basic();      results["Backtracking
básico"] = n1
_, n2 = backtracking_mrv_lcv();    results["BT + MRV
+ LCV"] = n2
_, n3 = backtracking_fc();         results["Forward
Checking"] = n3

fig, ax = plt.subplots(figsize=(9,5))
colors = ["#E24B4A","#EF9F27","#1D9E75"]
bars = ax.bar(results.keys(), results.values(), color=colors, edgecolor="#444", linewidth=0.5)
for bar, n in zip(bars, results.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            str(n), ha="center", va="bottom", fontweight="bold", fontsize=16)
ax.set_ylabel("Nodos explorados"); ax.set_ylim(0, max(results.values())*1.3)
ax.set_title("Comparación de estrategias CSP — Mapa de Australia", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

print("Resumen:")
for s,n in results.items(): print(f"  {s.replace(chr(10),' '):25s}: {n} nodos")

## 9. Resumen

| Técnica | Qué hace | Cuándo usar |
|---|---|---|
| **Backtracking** | DFS + verificar restricciones | Base para todos los demás |
| **MRV** | Variable más restringida primero | Siempre (mejora BT gratis) |
| **LCV** | Valor menos restrictivo primero | Cuando hay muchas soluciones |
| **Forward Checking** | Eliminar valores inconsistentes tras asignar | Mejora sobre BT puro |
| **AC-3** | Propagación completa de arc consistency | Preprocesamiento o integrado (MAC) |
| **Min-Conflicts** | Búsqueda local iterativa | CSPs muy grandes |

### Ideas clave:
- **CSP explota la estructura del estado**: variables + dominios + restricciones permiten podar inteligentemente
- **Fail-first para variables** (MRV): detectar fallos temprano
- **Fail-last para valores** (LCV): maximizar chances de éxito
- **Propagación** (FC, AC-3): eliminar imposibles antes de llegar a ellos
- **Backtracking es DFS con restricciones**: la conexión entre búsqueda y CSP